# 2. Local Debugging Tutor

### Build a small local debugging tutor and compare how prompt text and recent chat share a limited context window.

A local model can only read the text that fits inside one request.
That space is the **context window**.
The system prompt, the few-shot example, recent chat turns, and the new user message all use that same space.

This notebook keeps the model fixed and changes how that space is used.
One run removes earlier chat.
Another run keeps a short sliding window of recent turns.
The few-shot example is a second prompt control, so it also uses part of the same context budget.

**Focus**
- how the system prompt sets the tutor's role and limits
- how the few-shot example changes reply style
- how much room is left for chat history after extra prompt text is added
- how the next reply changes when recent turns are kept or removed



In [8]:
# %pip -q install gradio llama-cpp-python

import os  # lets us work with file paths and file names
import time  # lets us measure response time in seconds
import gradio as gr  # builds a simple web interface
from llama_cpp import Llama  # runs a local GGUF language model

## 2. Load a small local model

A local model runs on the machine directly instead of calling an external API.
In this notebook, the model file is a **GGUF** file, which is a format commonly used with `llama-cpp-python`.
Loading the model means reading that file and creating a model object that can answer prompts inside the notebook.

This cell does two jobs:
1. point to the model file
2. load the model into memory

It also defines `context_window`.
A context window is the total amount of text the model can read in one request.
That total includes every prompt instruction and every chat turn sent with the request.
If more text is added, more of the limited space is used.
A larger window allows more text to fit at once, but it also uses more memory during inference.

**What to expect after running this cell**
A model name should print at the end.
That output confirms that the notebook can find the GGUF file and create the local model object.
If the model name does not print, the path or the load settings need attention before moving on.


In [9]:
# Change this to match where your local GGUF model is stored.
# model_file_path = "/home/jovyan/shared/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"
# model_file_path = "/home/jovyan/shared/Qwen2.5-3B-Instruct-Q4_K_M.gguf"
model_file_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"
# model_file_path = "/Users/seohachoi/Downloads/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"

# Keep the context window in its own variable so we can reuse it later in the UI badge.
context_window = 1024

local_model = Llama(
    model_path=model_file_path,
    n_gpu_layers=0, # how much GPU to use (-1 = max, 0 = CPU only) | if unstable 24 -> 20 -> 16 -> 12 -> 10
    n_batch=64, # work size per step (bigger = faster, more memory) | if unstable, 32 -> 16 -> 8
    n_ctx=context_window, # context window (bigger = longer context, more memory) | if unstable, 1024 -> 512
    n_threads=4, # CPU thread count (usually 2–4 is safe)
    verbose=False,
)

# Extract the filename for UI
local_model_name = os.path.basename(model_file_path).replace(".gguf", "")
print("Model loaded:", local_model_name)

llama_context: n_ctx_per_seq (1024) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Model loaded: qwen2-1_5b-instruct-q4_0


## 3. Blueprint

Before the app runs, the tutor needs a prompt and a small set of repeatable questions.

- `system_prompt` sets the tutor's role and tone
- `few_shot_example` shows one example of the reply style
- `test_cases` stores one short starter prompt and two follow-up details for each case

The dropdown loads only the starter prompt.
That first message is intentionally short.
The `Next Turn` button adds `detail_1` and then `detail_2`.
That makes the workflow closer to a real debugging conversation, where the question becomes more specific after the first reply.

These pieces all use the same context window.
The context window is not reserved only for chat history.
Prompt instructions, examples, recent turns, and the new user message must all fit inside the same token budget.

The main comparison in this notebook is simple.

- `Memory Turns = 0` sends no earlier chat
- `Memory Turns = 1` or `2` keeps a short sliding window of recent turns

`Use Few-Shot` is a second prompt control.
It changes the style of the reply, and it also uses part of the same context space.




In [10]:
app_title = "AI Debugging Tutor — Small Local Model"
app_desc = (
    "A local debugging tutor with few-shot prompting, "
    "sliding-window memory, and a simple experiment log."
)

system_prompt = '''
You are a debugging tutor.

Rules:
- Do not give the final answer or full corrected code.
- Ask one short question or suggest one short check.
- Keep each reply short and focused.
- Stay on one issue at a time.
'''

few_shot_example = '''
Example:
User:
result = 85
print(result)

This failed one grader test.

Assistant:
What exact output was the grader expecting?
'''

# Each case starts with a short bug report.
# Add detail_1 and detail_2 in later turns.
test_cases = {
    "1. printed value failed the test [Easy]": {
        "starter": """scores = [78, 85, 92]
average_score = sum(scores) / len(scores)
print(int(average_score))

This failed one grader test.""",
        "detail_1": "The expected output is 85.0.",
        "detail_2": "The grader says the value is right, but the printed format is wrong."
    },
    "2. table columns failed the test [Easy]": {
        "starter": """cookie_summary = Table().with_columns(
    'Type', ['Chocolate chip', 'Red velvet'],
    'Amount remaining', [15, 8]
).select('Type', 'Amount remaining')
print(cookie_summary)

I thought this table was fine, but it still failed.""",
        "detail_1": "The expected column order is Amount remaining, Type.",
        "detail_2": "The grader says the values are correct and only the column order is wrong."
    },
    "3. sample changed between runs [Mid]": {
        "starter": """student_scores = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105, 106],
    'score': [88, 91, 75, 84, 93, 79]
})

first_sample = student_scores.sample(n=3)
second_sample = student_scores.sample(n=3)

print(first_sample)
print()
print(second_sample)

The sample changes every time I run this.""",
        "detail_1": "I need the same rows each time.",
        "detail_2": "The hidden test says the result is not reproducible."
    },
    "4. filtered rows still failed [Mid]": {
        "starter": """student_scores = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105],
    'score': [88, 91, 75, 84, 93]
})

top_scores = student_scores[student_scores['score'] >= 90]
print(top_scores)

These look like the right rows, but the test still fails.""",
        "detail_1": "The printed index is 1 and 4.",
        "detail_2": "The expected index is 0 and 1."
    },
    "5. grouped summary lost one category [Hard]": {
        "starter": """attendance = pd.DataFrame({
    'section': ['A', 'A', None, 'B'],
    'present': [1, 0, 1, 1]
})

attendance_rate = attendance.groupby('section')['present'].mean()
print(attendance_rate)

One group disappeared from the result.""",
        "detail_1": "The missing rows have blank values in the section column.",
        "detail_2": "The hidden test expects the blank section values to appear as their own group."
    },
}



## 4. State

A chat app needs a small amount of information that stays available while the app is open.
That stored information is often called **state**.
State is not part of the model itself.
It is data that the notebook keeps so later code cells and later button clicks can use it.

This notebook keeps one main state variable: `run_log`.
The log stores the recent settings and measurements from each run.
That makes it possible to compare prompt choices, memory settings, token counts, and latency across repeated tests instead of relying on memory alone.


In [11]:
run_log = []


## 5. Helper functions

This section contains the logic that prepares messages, counts tokens, runs the local model, and records results.
Although the code is grouped into functions, the job of the section is still concrete: build the exact prompt, send it to the model, and store the measurements.

Two panels make that logic easier to inspect.

- `Context` shows the exact message list sent to the model
- `Memory Window` shows which earlier turns stayed inside the sliding window

These panels matter because a local model does not keep conversation history automatically.
Every message that should influence the next reply has to be added to the prompt explicitly.
They also make the context budget visible by showing how prompt instructions, few-shot text, and recent turns are packed into one request.
When the reply changes, these panels make it easier to trace that change back to the exact input the model saw.


In [12]:
def clean_message(message):
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: ")[0]

    return {"role": role, "content": text}


def split_history(history, memory_turns):
    keep_messages = int(memory_turns) * 2

    if keep_messages <= 0:
        return []

    if len(history) <= keep_messages:
        return list(history)

    return list(history[-keep_messages:])


def count_text_tokens(text):
    if local_model is not None:
        try:
            return len(local_model.tokenize(text.encode("utf-8")))
        except Exception:
            pass
    return max(1, len(str(text).split()))


def build_messages(system_prompt, few_shot_example, use_few_shot, history, memory_turns):
    prompt_text = system_prompt.strip()
    if use_few_shot:
        prompt_text += "\n\n" + few_shot_example.strip()

    messages = [{"role": "system", "content": prompt_text}]
    recent_history = split_history(history, memory_turns)

    for message in recent_history:
        messages.append(clean_message(message))

    return messages


def build_context_text(messages):
    parts = []
    for message in messages:
        parts.append(f"[{message['role'].upper()}]")
        parts.append(str(message["content"]))
        parts.append("-" * 40)
    return "\n".join(parts)


def build_memory_window_text(history, memory_turns):
    keep_messages = int(memory_turns) * 2
    total_messages = len(history)

    if keep_messages <= 0:
        return "Memory is off. Only the current user message is sent."

    recent_history = split_history(history, memory_turns)
    dropped_messages = max(0, total_messages - len(recent_history))

    lines = []
    lines.append("Sliding window keeps the most recent messages only.")
    lines.append("Messages currently inside the window: " + str(len(recent_history)))
    lines.append("Messages outside the window: " + str(dropped_messages))
    lines.append("")

    if len(recent_history) == 0:
        lines.append("No earlier messages are inside the window yet.")
    else:
        lines.append("Current window contents:")
        for message in recent_history:
            clean_item = clean_message(message)
            lines.append(f"[{clean_item['role'].upper()}] {clean_item['content']}")

    return "\n".join(lines)


def generate_response(messages, temperature, max_tokens):
    if local_model is None:
        final_text = (
            "No local model is available. Update `model_file_path`, rerun the load cell, "
            "and then try again."
        )
        return final_text, None, count_text_tokens(final_text)

    try:
        response = local_model.create_chat_completion(
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens),
            top_p=0.95,
        )
        final_text = response["choices"][0]["message"]["content"] or ""
        usage = response.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens")
        output_tokens = usage.get("completion_tokens")
        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)
        return final_text, prompt_tokens, output_tokens
    except Exception as error:
        final_text = "Generation failed: " + str(error)
        return final_text, None, count_text_tokens(final_text)


def run_chatbot(user_input, system_prompt, few_shot_example, use_few_shot, memory_turns, temperature, max_tokens, test_case, history):
    global run_log

    if history is None:
        history = []

    messages = build_messages(system_prompt, few_shot_example, use_few_shot, history, memory_turns)

    if len(history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip():
            user_message = user_input.strip()
        else:
            user_message = test_cases[test_case]["starter"]
    else:
        user_message = user_input.strip()

    if user_message == "":
        log_rows = []
        for record in run_log[-10:]:
            log_rows.append([
                record["run"],
                record["test_case"],
                record["few_shot"],
                record["memory_turns"],
                record["model"],
                record["input_tokens"],
                record["output_tokens"],
                record["latency"],
            ])
        memory_text = build_memory_window_text(history, memory_turns)
        return history, "", memory_text, log_rows, "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_message})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["starter"]
        else:
            display_user = "[" + test_case + "]"

    history = list(history)
    history.append({"role": "user", "content": display_user})
    history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, prompt_tokens, output_tokens = generate_response(messages, temperature, max_tokens)
    latency = round(time.perf_counter() - start_time, 2)

    input_tokens = prompt_tokens
    if input_tokens is None:
        input_tokens = count_text_tokens(context_text)

    model_label = local_model_name or "local-model-not-loaded"
    badge = (
        f"\n\n`Time: {latency}s"
        f" | Context: {round((input_tokens / max(1, context_window)) * 100, 1)}%"
        f" | Input tokens: {input_tokens}"
        f" | Output tokens: {output_tokens}`"
    )
    history[-1]["content"] = final_text + badge

    run_log.append({
        "run": len(run_log) + 1,
        "test_case": test_case,
        "few_shot": "On" if use_few_shot else "Off",
        "memory_turns": int(memory_turns),
        "model": model_label,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
    })

    log_rows = []
    for record in run_log[-10:]:
        log_rows.append([
            record["run"],
            record["test_case"],
            record["few_shot"],
            record["memory_turns"],
            record["model"],
            record["input_tokens"],
            record["output_tokens"],
            record["latency"],
        ])

    memory_text = build_memory_window_text(history, memory_turns)
    return history, context_text, memory_text, log_rows, "", "**Status:** Done"


def clear_chat():
    global run_log
    run_log = []
    return [], "", "Memory window is empty. Start chatting to fill it.", [], "", "**Status:** Ready"


def load_test_case(test_case):
    if test_case == "Free Typing":
        return "", "**Status:** Ready", 0
    return test_cases[test_case]["starter"], "**Status:** Loaded starter prompt for " + test_case, 0


def load_next_turn(test_case, detail_stage, current_text):
    if test_case == "Free Typing" or test_case not in test_cases:
        return current_text, "**Status:** Next Turn is only available for a test case.", detail_stage

    if detail_stage <= 0:
        next_text = test_cases[test_case]["detail_1"]
        return next_text, "**Status:** Loaded detail 1 for " + test_case, 1

    if detail_stage == 1:
        next_text = test_cases[test_case]["detail_2"]
        return next_text, "**Status:** Loaded detail 2 for " + test_case, 2

    return current_text, "**Status:** No more saved details for " + test_case, detail_stage


def run_chatbot_ui(user_input, system_prompt, few_shot_example, use_few_shot, memory_turns, temperature, max_tokens, test_case, chat_state):
    history, context_text, history_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        few_shot_example,
        use_few_shot,
        memory_turns,
        temperature,
        max_tokens,
        test_case,
        chat_state,
    )
    return history, context_text, history_text, log_rows, next_input, status_message, history


def clear_chat_ui():
    history, context_text, history_text, log_rows, next_input, status_message = clear_chat()
    return history, context_text, history_text, log_rows, next_input, status_message, history, 0





## 6. Build the interface

This section puts the tutor into a Gradio layout.
The layout is not only for convenience.
It is also part of the lesson because it keeps the important hidden information visible during each run.

The left side is the chat area.
The right side holds the controls.
The bottom area shows the hidden parts of the experiment: the exact context, the current memory window, and the recent log.

That layout makes three prompt layers visible at the same time.
The chat window shows the reply.
The context panel shows what text was actually sent.
The log shows how the token budget changed from one run to the next.
This makes the notebook easier to study because the output is shown together with the prompt choices that produced it.


In [19]:
ui_test_cases = ["Free Typing"] + list(test_cases.keys())

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")
    gr.Markdown(
        "**Loaded model:** " + (local_model_name if local_model_name else "No local model loaded yet")
    )

    chat_state = gr.State([])
    detail_stage_state = gr.State(0)

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(label="Chat", height=470)
            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your debugging question here...",
                    lines=9,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    next_turn_button = gr.Button("Next Turn", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_test_cases,
                value="Free Typing",
                label="Test Case",
            )

            use_few_shot = gr.Checkbox(label="Use Few-Shot", value=False)

            with gr.Accordion("Model Settings", open=True):
                memory_turns_slider = gr.Slider(0, 8, step=1, value=1, label="Memory Turns")
                temperature_slider = gr.Slider(0.0, 1.0, step=0.1, value=0.2, label="Temperature")
                max_tokens_slider = gr.Slider(64, 1024, step=64, value=384, label="Max Tokens")

            with gr.Accordion("Edit Prompts", open=False):
                system_prompt_box = gr.Textbox(label="System Prompt", value=system_prompt, lines=8)
                few_shot_box = gr.Textbox(label="Few-Shot Example", value=few_shot_example, lines=8)

    with gr.Row():
        context_box = gr.Textbox(label="Context", value="", lines=16, interactive=False)
        history_box = gr.Textbox(
            label="Memory Window",
            value="Memory window is empty. Start chatting to fill it.",
            lines=16,
            interactive=False,
        )

    log_table = gr.Dataframe(
        headers=["Run", "Test Case", "Few-Shot", "Memory Turns", "Model", "Input", "Output", "Latency"],
        datatype=["number", "str", "str", "number", "str", "number", "number", "number"],
        row_count=10,
        column_count=(8, "fixed"),
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    inputs = [
        user_input_box,
        system_prompt_box,
        few_shot_box,
        use_few_shot,
        memory_turns_slider,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        chat_state,
    ]

    outputs = [
        chat_window,
        context_box,
        history_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    test_case_dropdown.change(load_test_case, test_case_dropdown, [user_input_box, status_box, detail_stage_state])
    next_turn_button.click(load_next_turn, [test_case_dropdown, detail_stage_state, user_input_box], [user_input_box, status_box, detail_stage_state])
    ask_button.click(run_chatbot_ui, inputs, outputs)
    user_input_box.submit(run_chatbot_ui, inputs, outputs)
    clear_button.click(clear_chat_ui, None, outputs + [detail_stage_state])



## 7. Run the app

This cell launches the local web app.

Work with one case for several turns instead of changing cases too quickly.
The dropdown loads only the short starter prompt.
The same test case has two extra details that the `Next Turn` button can add later.

A useful workflow is:

1. Pick one test case and run the starter prompt.
2. Read the tutor reply and note what is still vague.
3. Click `Next Turn` once to add `detail_1`.
4. Compare the second reply with the first one.
5. Click `Next Turn` again to add `detail_2`.
6. Repeat the same case with `Memory Turns` set to `0`, then `1` or `2`.
7. Turn `Use Few-Shot` on after the memory comparison is clear.

**Look for**
- whether each added detail makes the next hint more concrete
- whether the tutor keeps the earlier bug description in later turns
- how the `Context` panel changes when more prompt text is added
- whether few-shot changes the tutoring style while using some of the same context space
- whether the token count grows as more prompt material is included




In [18]:
web_app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://92528551f973c11b4d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Summary

This notebook shows that a context window is a shared budget.
The system prompt, few-shot example, recent chat turns, and new user message all compete for the same space.
A local model only sees what is placed inside that request.
Anything left out is invisible on the next turn.

A sliding window is one way to manage that budget.
It keeps the most recent turns and leaves out older ones.
The few-shot example adds a second lesson: prompt design also uses part of the same limited space.

The main topic here is context management.
The notebook shows how prompt parts are chosen, how they fit inside the available window, and how those choices change the next reply.
